In [3]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth

# 학습 모델 저장을 위한 라이브러리
import pickle

# 폴더에 들어있는 파일 가져오기
import glob
import os

# 시간을 관리하는 모듈 
from datetime import datetime

# 예쁘게 출력하는 모듈
from IPython.display import display

#학습 모델 저장
import joblib

## 📤 데이터를 불러오고 학습/검증한다.

In [3]:
# 데이터 로딩
train = pd.read_csv("C:/Users/user/Desktop/workspace/14_Final_PROJECT/df_selected_only_50.csv")

# 피처 / 타겟 분리
X = train.drop(columns=["기준년월", "Segment"])
y = train["Segment"]

# 라벨 인코딩
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 스케일링 + PCA (95% 분산 유지)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95, random_state=1)
X_pca = pca.fit_transform(X_scaled)
print(f"✅ PCA 이후 차원 수: {X_pca.shape[1]}")

# 모델 정의
models = {
    "RandomForest": RandomForestClassifier(random_state=1),
    "LightGBM": LGBMClassifier(random_state=1),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=1)
}

# 교차검증 + Soft Voting + 성능 측정
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
f1_macro_list, f1_micro_list, acc_list = [], [], []

print("\n🚀 Soft Voting 앙상블 검증 시작")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_pca, y_encoded), 1):
    X_train, X_val = X_pca[train_idx], X_pca[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # 모델별 확률 예측 저장
    probs = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)
        probs.append(prob)

    # Soft Voting: 평균 확률
    avg_prob = np.mean(probs, axis=0)
    preds = np.argmax(avg_prob, axis=1)

    # 평가
    f1_macro = f1_score(y_val, preds, average='macro')
    f1_micro = f1_score(y_val, preds, average='micro')
    acc = accuracy_score(y_val, preds)

    f1_macro_list.append(f1_macro)
    f1_micro_list.append(f1_micro)
    acc_list.append(acc)

    print(f"📂 Fold {fold}: F1 Macro = {f1_macro:.4f}, F1 Micro = {f1_micro:.4f}, Accuracy = {acc:.4f}")

# 평균 결과 출력
print("\n📊 최종 평균 평가 결과 (5-Fold Soft Voting 앙상블):")
print(f"- F1 Macro  평균: {np.mean(f1_macro_list):.4f}")
print(f"- F1 Micro  평균: {np.mean(f1_micro_list):.4f}")
print(f"- Accuracy 평균: {np.mean(acc_list):.4f}")

✅ PCA 이후 차원 수: 42

🚀 Soft Voting 앙상블 검증 시작
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.376456 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10710
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 42
[LightGBM] [Info] Start training from score -7.812395
[LightGBM] [Info] Start training from score -9.714246
[LightGBM] [Info] Start training from score -2.934402
[LightGBM] [Info] Start training from score -1.927461
[LightGBM] [Info] Start training from score -0.222075
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]